# Working with Calibration Data

Calibration data is the bridge between your circuit and the actual hardware.
qb-compiler uses calibration snapshots to make noise-aware routing decisions,
estimate fidelity, and select the best qubits.

This notebook covers:
- `CalibrationProvider` hierarchy: Static, Live, and Cached providers
- Loading calibration snapshots from JSON files
- Inspecting `QubitProperties` (T1, T2, readout error, frequency)
- Inspecting `GateProperties` (error rates, gate times)
- Finding the best qubits on a backend
- Tracking calibration drift over time
- Calibration age and staleness detection

In [ ]:
from qb_compiler.calibration.static_provider import StaticCalibrationProvider
from qb_compiler.calibration.cached_provider import CachedCalibrationProvider
from qb_compiler.calibration.provider import CalibrationProvider
from qb_compiler.calibration.models.qubit_properties import QubitProperties
from qb_compiler.calibration.models.coupling_properties import GateProperties
from qb_compiler.calibration.models.backend_properties import BackendProperties

## 1. Loading a Calibration Snapshot

`StaticCalibrationProvider.from_json()` loads a QubitBoost calibration
hub JSON file into an in-memory provider. No network calls, no caching --
just a direct snapshot.

In [ ]:
# Load a recent IBM Fez calibration snapshot
provider = StaticCalibrationProvider.from_json(
    "../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json"
)

print(f"Provider: {provider}")
print(f"Backend:  {provider.backend_name}")
print(f"Taken at: {provider.timestamp}")
print(f"Age:      {provider.age_hours:.1f} hours")

# Access the underlying BackendProperties for full metadata
props = provider.properties
print(f"\nN qubits:    {props.n_qubits}")
print(f"Basis gates: {props.basis_gates}")
print(f"Coupling map edges: {len(props.coupling_map)}")

## 2. Inspecting QubitProperties

Each physical qubit has T1 (energy relaxation), T2 (dephasing),
readout errors, and drive frequency. These determine how "good" a
qubit is for computation.

In [ ]:
# Look at a specific qubit
qp = provider.get_qubit_properties(0)
if qp:
    print(f"Qubit 0:")
    print(f"  T1:              {qp.t1_us:.1f} us" if qp.t1_us else "  T1: N/A")
    print(f"  T2:              {qp.t2_us:.1f} us" if qp.t2_us else "  T2: N/A")
    print(f"  Readout error:   {qp.readout_error:.4f}" if qp.readout_error else "  Readout error: N/A")
    print(f"  Frequency:       {qp.frequency_ghz:.4f} GHz" if qp.frequency_ghz else "  Frequency: N/A")
    print(f"  P(1|0):          {qp.readout_error_0to1:.4f}" if qp.readout_error_0to1 is not None else "  P(1|0): N/A")
    print(f"  P(0|1):          {qp.readout_error_1to0:.4f}" if qp.readout_error_1to0 is not None else "  P(0|1): N/A")
    print(f"  T1 asymmetry:    {qp.t1_asymmetry_ratio:.2f}x")
    print(f"  Asymmetry penalty: {qp.t1_asymmetry_penalty:.4f}")

In [ ]:
# Summary of all qubits
all_qubits = provider.get_all_qubit_properties()
print(f"Total qubits: {len(all_qubits)}")

# Statistics
t1_values = [q.t1_us for q in all_qubits if q.t1_us is not None]
t2_values = [q.t2_us for q in all_qubits if q.t2_us is not None]
ro_values = [q.readout_error for q in all_qubits if q.readout_error is not None]

if t1_values:
    print(f"\nT1 (us):  min={min(t1_values):.1f}  median={sorted(t1_values)[len(t1_values)//2]:.1f}  max={max(t1_values):.1f}")
if t2_values:
    print(f"T2 (us):  min={min(t2_values):.1f}  median={sorted(t2_values)[len(t2_values)//2]:.1f}  max={max(t2_values):.1f}")
if ro_values:
    print(f"Readout:  min={min(ro_values):.4f}  median={sorted(ro_values)[len(ro_values)//2]:.4f}  max={max(ro_values):.4f}")

## 3. Inspecting GateProperties

Gate calibration data includes error rates (from randomised benchmarking)
and gate durations. Two-qubit gates are indexed by `(gate_type, (q0, q1))`.

In [ ]:
# Look at a specific gate
gp = provider.get_gate_properties("cx", (0, 1))
if gp:
    print(f"CX gate on qubits (0, 1):")
    print(f"  Error rate:  {gp.error_rate:.6f}" if gp.error_rate else "  Error rate: N/A")
    print(f"  Gate time:   {gp.gate_time_ns:.1f} ns" if gp.gate_time_ns else "  Gate time: N/A")
else:
    print("CX(0,1) not found -- qubits may not be directly connected.")
    # Try to find some valid CX gate
    all_gates = provider.get_all_gate_properties()
    cx_gates = [g for g in all_gates if g.gate_type == "cx" and g.error_rate is not None]
    if cx_gates:
        sample = cx_gates[0]
        print(f"Example: CX{sample.qubits} error={sample.error_rate:.6f}")

In [ ]:
# Gate error statistics
all_gates = provider.get_all_gate_properties()
print(f"Total gate entries: {len(all_gates)}")

# Group by gate type
from collections import defaultdict
gate_groups = defaultdict(list)
for gp in all_gates:
    if gp.error_rate is not None:
        gate_groups[gp.gate_type].append(gp.error_rate)

print(f"\n{'Gate type':<8} {'Count':>6} {'Min err':>10} {'Median err':>12} {'Max err':>10}")
print("-" * 50)
for gate_type, errors in sorted(gate_groups.items()):
    errors.sort()
    median = errors[len(errors) // 2]
    print(
        f"{gate_type:<8} {len(errors):>6} "
        f"{min(errors):>10.6f} {median:>12.6f} {max(errors):>10.6f}"
    )

## 4. Finding the Best Qubits

Not all qubits are equal. Some have much better T1/T2 and readout fidelity.
Let's rank them to find the best ones for our circuits.

In [ ]:
def score_qubit(qp):
    """Score a qubit -- lower is better.

    Combines readout error, inverse T1, and inverse T2.
    """
    ro = qp.readout_error if qp.readout_error is not None else 0.05
    t1 = qp.t1_us if qp.t1_us is not None else 50.0
    t2 = qp.t2_us if qp.t2_us is not None else 25.0
    return 0.5 * ro + 0.25 * (1.0 / max(t1, 1.0)) + 0.25 * (1.0 / max(t2, 1.0))


all_qubits = provider.get_all_qubit_properties()
scored = [(qp, score_qubit(qp)) for qp in all_qubits]
scored.sort(key=lambda x: x[1])

print("Top 10 best qubits:")
print(f"{'Rank':>4} {'Qubit':>6} {'T1 (us)':>8} {'T2 (us)':>8} {'RO err':>8} {'Score':>8}")
print("-" * 44)
for rank, (qp, sc) in enumerate(scored[:10], 1):
    t1 = f"{qp.t1_us:.0f}" if qp.t1_us else "N/A"
    t2 = f"{qp.t2_us:.0f}" if qp.t2_us else "N/A"
    ro = f"{qp.readout_error:.4f}" if qp.readout_error else "N/A"
    print(f"{rank:>4} {qp.qubit_id:>6} {t1:>8} {t2:>8} {ro:>8} {sc:>8.5f}")

print(f"\nWorst 5 qubits:")
for rank, (qp, sc) in enumerate(scored[-5:], len(scored) - 4):
    t1 = f"{qp.t1_us:.0f}" if qp.t1_us else "N/A"
    ro = f"{qp.readout_error:.4f}" if qp.readout_error else "N/A"
    print(f"{rank:>4} {qp.qubit_id:>6} {t1:>8}  ro={ro}  score={sc:.5f}")

## 5. Tracking Calibration Drift Over Time

Calibration data changes as the hardware drifts. Let's compare two
snapshots from different days to see how much things move.

In [ ]:
# Load two snapshots from different dates
provider_old = StaticCalibrationProvider.from_json(
    "../tests/fixtures/calibration_snapshots/ibm_fez_2026_02_15.json"
)
provider_new = StaticCalibrationProvider.from_json(
    "../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json"
)

print(f"Old snapshot: {provider_old.timestamp}")
print(f"New snapshot: {provider_new.timestamp}")

# Compare readout errors
drifts = []
for qp_old in provider_old.get_all_qubit_properties():
    qp_new = provider_new.get_qubit_properties(qp_old.qubit_id)
    if qp_new and qp_old.readout_error is not None and qp_new.readout_error is not None:
        delta = qp_new.readout_error - qp_old.readout_error
        drifts.append((qp_old.qubit_id, delta, qp_old.readout_error, qp_new.readout_error))

drifts.sort(key=lambda x: abs(x[1]), reverse=True)

print(f"\nReadout error drift (top 10 biggest changes):")
print(f"{'Qubit':>6} {'Old':>10} {'New':>10} {'Delta':>10}")
print("-" * 38)
for qid, delta, old_ro, new_ro in drifts[:10]:
    direction = "+" if delta > 0 else ""
    print(f"{qid:>6} {old_ro:>10.5f} {new_ro:>10.5f} {direction}{delta:>9.5f}")

avg_drift = sum(abs(d[1]) for d in drifts) / len(drifts) if drifts else 0
print(f"\nAverage |drift|: {avg_drift:.5f}")

In [ ]:
# Compare T1 drift
t1_drifts = []
for qp_old in provider_old.get_all_qubit_properties():
    qp_new = provider_new.get_qubit_properties(qp_old.qubit_id)
    if qp_new and qp_old.t1_us is not None and qp_new.t1_us is not None:
        pct_change = ((qp_new.t1_us - qp_old.t1_us) / qp_old.t1_us) * 100
        t1_drifts.append((qp_old.qubit_id, qp_old.t1_us, qp_new.t1_us, pct_change))

t1_drifts.sort(key=lambda x: abs(x[3]), reverse=True)

print(f"T1 drift (top 10 biggest % changes):")
print(f"{'Qubit':>6} {'Old T1':>8} {'New T1':>8} {'Change':>8}")
print("-" * 34)
for qid, old_t1, new_t1, pct in t1_drifts[:10]:
    print(f"{qid:>6} {old_t1:>8.1f} {new_t1:>8.1f} {pct:>+7.1f}%")

## 6. CachedCalibrationProvider for Production

In production, you want calibration data that auto-refreshes.
`CachedCalibrationProvider` wraps any provider and re-fetches when
the cached data gets stale.

In [ ]:
# For demonstration, wrap a static provider in a caching layer.
# In production, you'd wrap a LiveCalibrationProvider.

call_count = 0

def factory():
    """Simulates fetching fresh calibration."""
    global call_count
    call_count += 1
    print(f"  [factory called: fetch #{call_count}]")
    return StaticCalibrationProvider.from_json(
        "../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json"
    )


cached = CachedCalibrationProvider(
    provider_factory=factory,
    max_age_seconds=3600,        # refresh after 1 hour
    hard_limit_hours=24.0,       # reject data older than 24h
)

print("First access (triggers fetch):")
qp = cached.get_qubit_properties(0)
print(f"  Qubit 0 T1: {qp.t1_us:.1f} us" if qp and qp.t1_us else "  N/A")

print("\nSecond access (served from cache):")
qp = cached.get_qubit_properties(1)
print(f"  Qubit 1 T1: {qp.t1_us:.1f} us" if qp and qp.t1_us else "  N/A")

print(f"\nTotal factory calls: {call_count}")

In [ ]:
# Manual cache control
print("Before invalidate: factory won't be called")
_ = cached.get_qubit_properties(0)

print("\nAfter invalidate: forces re-fetch")
cached.invalidate()
_ = cached.get_qubit_properties(0)

print(f"\nTotal factory calls: {call_count}")

## 7. Calibration Age and Staleness

Every `CalibrationProvider` exposes `age_hours` -- the time elapsed
since the calibration was taken. Stale data leads to suboptimal
qubit selection.

In [ ]:
# Check ages of several snapshots
import os
import glob

files = sorted(glob.glob("../tests/fixtures/calibration_snapshots/ibm_fez_*.json"))

print(f"{'File':<50} {'Timestamp':<26} {'Age (hrs)':>10}")
print("-" * 88)
for f in files:
    try:
        p = StaticCalibrationProvider.from_json(f)
        print(f"{os.path.basename(f):<50} {str(p.timestamp):<26} {p.age_hours:>10.1f}")
    except Exception as e:
        print(f"{os.path.basename(f):<50} Error: {e}")

## 8. Calibration Quality Heatmap (Text-Based)

A quick way to visualise qubit quality across the device.
We assign quality grades based on readout error thresholds.

In [ ]:
provider = StaticCalibrationProvider.from_json(
    "../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json"
)

def quality_grade(qp):
    """Assign a quality grade based on readout error."""
    if qp.readout_error is None:
        return "?"
    ro = qp.readout_error
    if ro < 0.005:
        return "A"  # Excellent
    elif ro < 0.01:
        return "B"  # Good
    elif ro < 0.02:
        return "C"  # Acceptable
    elif ro < 0.05:
        return "D"  # Poor
    else:
        return "F"  # Bad


all_qubits = provider.get_all_qubit_properties()
grades = {qp.qubit_id: quality_grade(qp) for qp in all_qubits}

# Grade distribution
from collections import Counter
dist = Counter(grades.values())

print(f"Qubit quality distribution for {provider.backend_name}:")
print(f"  A (ro < 0.5%):  {dist.get('A', 0):>3} qubits")
print(f"  B (ro < 1.0%):  {dist.get('B', 0):>3} qubits")
print(f"  C (ro < 2.0%):  {dist.get('C', 0):>3} qubits")
print(f"  D (ro < 5.0%):  {dist.get('D', 0):>3} qubits")
print(f"  F (ro >= 5.0%): {dist.get('F', 0):>3} qubits")
print(f"  ? (no data):    {dist.get('?', 0):>3} qubits")

# Print first 40 qubits as a compact text heatmap
n_show = min(40, len(grades))
print(f"\nFirst {n_show} qubits: ", end="")
for q in range(n_show):
    print(grades.get(q, "."), end="")
print()
print("Legend: A=excellent  B=good  C=ok  D=poor  F=bad")

## 9. The CalibrationProvider Hierarchy

qb-compiler provides three provider types for different use cases:

| Provider | Network | Caching | Use case |
|----------|---------|---------|----------|
| `StaticCalibrationProvider` | No | No | Offline analysis, testing, replays |
| `LiveCalibrationProvider` | Yes | Built-in | Production (requires `qubitboost-sdk`) |
| `CachedCalibrationProvider` | Via factory | Thread-safe | Production wrapper for any provider |

All implement the same `CalibrationProvider` abstract interface:
- `get_qubit_properties(qubit)` -- single qubit lookup
- `get_gate_properties(gate, qubits)` -- single gate lookup
- `get_all_qubit_properties()` -- bulk qubit data
- `get_all_gate_properties()` -- bulk gate data
- `backend_name` -- which backend this data is for
- `timestamp` -- when the data was taken
- `age_hours` -- how old the data is

In [ ]:
# You can also build a StaticCalibrationProvider from a dict
# (useful when you have calibration data from an API response)

cal_dict = {
    "backend_name": "custom_device",
    "n_qubits": 3,
    "timestamp": "2026-03-17T12:00:00Z",
    "basis_gates": ["cx", "rz", "sx", "x"],
    "coupling_map": [[0, 1], [1, 2]],
    "qubit_properties": [
        {"qubit": 0, "T1": 250.0, "T2": 120.0, "frequency": 5.1,
         "readout_error_0to1": 0.005, "readout_error_1to0": 0.015},
        {"qubit": 1, "T1": 300.0, "T2": 150.0, "frequency": 5.2,
         "readout_error_0to1": 0.003, "readout_error_1to0": 0.012},
        {"qubit": 2, "T1": 200.0, "T2": 100.0, "frequency": 5.0,
         "readout_error_0to1": 0.008, "readout_error_1to0": 0.020},
    ],
    "gate_properties": [
        {"gate": "cx", "qubits": [0, 1], "parameters": {"gate_error": 0.005, "gate_length": 400.0}},
        {"gate": "cx", "qubits": [1, 2], "parameters": {"gate_error": 0.007, "gate_length": 420.0}},
    ],
}

custom = StaticCalibrationProvider.from_dict(cal_dict)
print(f"Custom provider: {custom}")
print(f"Qubit 1 T1: {custom.get_qubit_properties(1).t1_us} us")
print(f"CX(0,1) error: {custom.get_gate_properties('cx', (0, 1)).error_rate}")

## Summary

Key takeaways:

- **Always use fresh calibration data** -- quantum hardware drifts hourly
- **Check `age_hours`** before trusting results; > 12 hours is risky
- **Not all qubits are equal** -- scoring and ranking helps pick the best subset
- **`CachedCalibrationProvider`** handles auto-refresh in production
- **`StaticCalibrationProvider.from_json()`** is the easiest way to start

The calibration data flows through the entire qb-compiler pipeline:
calibration -> noise model -> fidelity estimation -> routing decisions.